# Classical Machine Learning Baselines

This notebook establishes the classical machine-learning baselines for the
UNSW-NB15 binary intrusion-detection task.

The previous preprocessing experiments produced multiple candidate feature
representations designed to address differences in feature scale, skewness,
outliers, and redundancy.

Rather than assuming that a particular preprocessing strategy is optimal, this
notebook evaluates the candidate preprocessing variants using classical
machine-learning models.

The objectives are to:

1. Establish strong classical baseline performance.
2. Compare the effect of different preprocessing strategies.
3. Identify which model-preprocessing combinations are most effective.
4. Establish a benchmark for the later constrained QML and hybrid-model stages.

All models will be evaluated using the same official training and testing
splits to ensure a consistent comparison.

## 1. Imports and Configuration

This section imports the libraries required for data loading, preprocessing,
model training, and evaluation.

In [11]:
#importing libraries 
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    RobustScaler,
    FunctionTransformer
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from src.data.preprocessing import UNSWPreprocessor

## 2. Project Path Configuration

The notebook is executed from the `notebooks/03_baseline_models` directory,
while the dataset and preprocessing utilities are located elsewhere in the
project structure.

The project root is therefore added to the Python path so that project modules
and data files can be accessed consistently.

In [2]:
#notebook config
PROJECT_ROOT = Path.cwd().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:")
print(PROJECT_ROOT)

Project root:
c:\Projects\Aeges-Q


## 3. Loading the Dataset

The official UNSW-NB15 training and testing splits are loaded for baseline
model development.

The original splits are preserved to maintain consistency with the previous
EDA and preprocessing experiments. No additional train-test split is created.

The `label` column is used as the binary target, while `attack_cat` is retained
only as metadata and is not used as a model input feature.

In [3]:
train_path = PROJECT_ROOT / "data" / "raw" / "UNSW_NB15_training-set.csv"
test_path = PROJECT_ROOT / "data" / "raw" / "UNSW_NB15_testing-set.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Training set shape:", train_df.shape)
print("Testing set shape:", test_df.shape)

Training set shape: (82332, 45)
Testing set shape: (175341, 45)


## 4. Feature and Target Preparation

The raw dataset contains an identifier column, the binary classification target,
and attack-category metadata.

To maintain consistency with the preprocessing experiments, the existing
`UNSWPreprocessor` utility is used to separate:

- Model input features (`X`)
- Binary target labels (`y`)
- Attack-category metadata

Only the binary `label` is used for the baseline classification experiments.
The `attack_cat` column is retained separately for potential later analysis.

In [5]:
preprocessor = UNSWPreprocessor()

X_train, y_train, attack_train = (
    preprocessor.split_features_and_target(train_df)
)

X_test, y_test, attack_test = (
    preprocessor.split_features_and_target(test_df)
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nX_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (82332, 42)
y_train: (82332,)

X_test: (175341, 42)
y_test: (175341,)


## 5. Evaluation Framework

All model-preprocessing combinations will be evaluated using the same set of
classification metrics.

Because intrusion detection involves identifying malicious traffic, accuracy
alone is insufficient. The evaluation therefore includes:

- **Accuracy:** Overall proportion of correctly classified samples.
- **Precision:** Proportion of predicted attacks that are actually attacks.
- **Recall:** Proportion of actual attacks correctly detected.
- **F1-score:** Harmonic mean of precision and recall.
- **ROC-AUC:** Ability of the model to distinguish between normal and attack
  traffic across different classification thresholds.

Using a common evaluation framework ensures that all preprocessing and model
combinations can be compared consistently.

In [7]:
def evaluate_model(model, X_test, y_test):

    y_pred = model.predict(X_test)

    y_proba = model.predict_proba(X_test)[:, 1]

    results = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba)
    }

    return results

## 6. Preprocessing Setup

This section reconstructs the preprocessing variants developed in the previous
notebook so they can be evaluated consistently across the baseline models.
Each transformer is fitted only on the training data before being applied to
the testing data.

In [9]:
preprocessor.identify_feature_types(X_train)

numerical_features = preprocessor.numerical_features
categorical_features = preprocessor.categorical_features

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical features:")
print(categorical_features)

Numerical features: 39
Categorical features: 3

Categorical features:
['proto', 'service', 'state']


In [10]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

In [12]:
# base numerical pipeline 
base_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

In [13]:
# standard-scaled numerical pipeline 
standard_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [18]:
# robust-scaled numerical pipeline
robust_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler())
    ]
)

In [19]:
# piepline 
preprocessing_variants = {}

preprocessing_variants["A"] = ColumnTransformer(
    transformers=[
        ("numeric", base_numeric_transformer, numerical_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

preprocessing_variants["B"] = ColumnTransformer(
    transformers=[
        ("numeric", standard_numeric_transformer, numerical_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

preprocessing_variants["C"] = ColumnTransformer(
    transformers=[
        ("numeric", robust_numeric_transformer, numerical_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

In [20]:
# D1-Broad Log Transform 
broad_log_features = [
    "dur",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
    "rate",
    "sload",
    "dload",
    "sloss",
    "dloss",
    "sinpkt",
    "dinpkt",
    "sjit",
    "djit",
    "tcprtt",
    "synack",
    "ackdat",
    "smean",
    "dmean",
    "trans_depth",
    "response_body_len",
    "ct_srv_src",
    "ct_state_ttl",
    "ct_dst_ltm",
    "ct_src_dport_ltm",
    "ct_dst_sport_ltm",
    "ct_dst_src_ltm",
    "is_ftp_login",
    "ct_ftp_cmd",
    "ct_flw_http_mthd",
    "ct_src_ltm",
    "ct_srv_dst",
    "is_sm_ips_ports"
]

remaining_d1_features = [
    feature
    for feature in numerical_features
    if feature not in broad_log_features
]

In [21]:
log_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p)),
        ("scaler", StandardScaler())
    ]
)

In [22]:
preprocessing_variants["D1"] = ColumnTransformer(
    transformers=[
        (
            "log_numeric",
            log_numeric_transformer,
            broad_log_features
        ),
        (
            "remaining_numeric",
            standard_numeric_transformer,
            remaining_d1_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [23]:
# selective log transformation
selective_log_features = [
    "dur",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
    "rate",
    "sload",
    "dload",
    "sloss",
    "dloss",
    "sinpkt",
    "dinpkt",
    "sjit",
    "djit",
    "tcprtt",
    "synack",
    "ackdat",
    "smean",
    "dmean",
    "trans_depth",
    "response_body_len",
    "ct_srv_src",
    "ct_state_ttl",
    "ct_dst_ltm",
    "ct_src_dport_ltm",
    "ct_dst_sport_ltm",
    "ct_dst_src_ltm",
    "ct_flw_http_mthd",
    "ct_src_ltm",
    "ct_srv_dst"
]

remaining_d2_features = [
    feature
    for feature in numerical_features
    if feature not in selective_log_features
]

In [24]:
preprocessing_variants["D2"] = ColumnTransformer(
    transformers=[
        (
            "log_numeric",
            log_numeric_transformer,
            selective_log_features
        ),
        (
            "remaining_numeric",
            standard_numeric_transformer,
            remaining_d2_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

In [26]:
# redundancy reduction
features_to_drop = [
    "ct_dst_ltm",
    "ct_dst_src_ltm",
    "ct_ftp_cmd",
    "ct_src_dport_ltm",
    "ct_src_ltm",
    "ct_srv_src",
    "dloss",
    "dpkts",
    "dwin",
    "is_sm_ips_ports",
    "sloss",
    "spkts",
    "tcprtt"
]

In [27]:
reduced_numerical_features = [
    feature
    for feature in numerical_features
    if feature not in features_to_drop
]

In [28]:
print("Original numerical features:", len(numerical_features))
print("Reduced numerical features:", len(reduced_numerical_features))

Original numerical features: 39
Reduced numerical features: 26


In [29]:
preprocessing_variants["E"] = ColumnTransformer(
    transformers=[
        (
            "numeric",
            standard_numeric_transformer,
            reduced_numerical_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

## 7. Generating Preprocessed Feature Representations

Each preprocessing variant is fitted exclusively on the training data and then
used to transform both the training and testing sets.

The resulting feature matrices are stored so that every baseline model can be
evaluated on exactly the same preprocessing variants.

In [31]:
processed_data = {}

In [32]:
for variant_name, transformer in preprocessing_variants.items():

    print(f"Processing Variant {variant_name}...")

    X_train_processed = transformer.fit_transform(X_train)

    X_test_processed = transformer.transform(X_test)

    processed_data[variant_name] = {
        "X_train": X_train_processed,
        "X_test": X_test_processed
    }

    print(
        "Train shape:",
        X_train_processed.shape
    )

    print(
        "Test shape:",
        X_test_processed.shape
    )

    print()

Processing Variant A...
Train shape: (82332, 190)
Test shape: (175341, 190)

Processing Variant B...
Train shape: (82332, 190)
Test shape: (175341, 190)

Processing Variant C...
Train shape: (82332, 190)
Test shape: (175341, 190)

Processing Variant D1...
Train shape: (82332, 190)
Test shape: (175341, 190)

Processing Variant D2...
Train shape: (82332, 190)
Test shape: (175341, 190)

Processing Variant E...
Train shape: (82332, 177)
Test shape: (175341, 177)



In [33]:
for variant_name, data in processed_data.items():

    print(
        f"{variant_name}:",
        "Train =", data["X_train"].shape,
        "| Test =", data["X_test"].shape
    )

A: Train = (82332, 190) | Test = (175341, 190)
B: Train = (82332, 190) | Test = (175341, 190)
C: Train = (82332, 190) | Test = (175341, 190)
D1: Train = (82332, 190) | Test = (175341, 190)
D2: Train = (82332, 190) | Test = (175341, 190)
E: Train = (82332, 177) | Test = (175341, 177)


## 8. Baseline Model Benchmarking

Each preprocessing variant is evaluated using the same classical machine-learning
models.

The initial baselines consist of:

- **Logistic Regression**, providing a linear baseline and allowing the effect
  of feature scaling and transformation to be examined.
- **Random Forest**, providing a nonlinear tree-based baseline that is less
  sensitive to feature scaling.

Each model-preprocessing combination is trained on the official training split
and evaluated on the official testing split using the common evaluation
framework defined earlier.

In [34]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
}

In [35]:
# benchmark loop 
benchmark_results = []

for variant_name, data in processed_data.items():

    X_train_variant = data["X_train"]
    X_test_variant = data["X_test"]

    for model_name, model in models.items():

        print(
            f"Training {model_name} "
            f"on Variant {variant_name}..."
        )

        model.fit(
            X_train_variant,
            y_train
        )

        results = evaluate_model(
            model,
            X_test_variant,
            y_test
        )

        results["variant"] = variant_name
        results["model"] = model_name

        benchmark_results.append(results)

        print(
            f"F1 Score: {results['f1_score']:.4f}"
        )

        print()

Training Logistic Regression on Variant A...


c:\Users\Askari\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


F1 Score: 0.7749

Training Random Forest on Variant A...
F1 Score: 0.9252

Training Logistic Regression on Variant B...
F1 Score: 0.9032

Training Random Forest on Variant B...
F1 Score: 0.9254

Training Logistic Regression on Variant C...


c:\Users\Askari\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


F1 Score: 0.8630

Training Random Forest on Variant C...
F1 Score: 0.9251

Training Logistic Regression on Variant D1...
F1 Score: 0.9150

Training Random Forest on Variant D1...
F1 Score: 0.9254

Training Logistic Regression on Variant D2...
F1 Score: 0.9150

Training Random Forest on Variant D2...
F1 Score: 0.9251

Training Logistic Regression on Variant E...
F1 Score: 0.9155

Training Random Forest on Variant E...
F1 Score: 0.9335

